# 2. Análisis exploratorio

La segunda pestaña del tablero reúne el análisis exploratorio en siete secciones
desplegables. Este capítulo sigue el mismo orden, reproduce cada figura y tabla a partir de
la serie diaria de caudales y expone las decisiones de modelado que se derivan de cada
resultado. Todos los cálculos que informan esas decisiones se restringen al periodo de
entrenamiento, anterior al 1 de enero de 2010, de modo que el periodo de prueba permanece
intacto hasta la evaluación del capítulo 3.

In [1]:
import json
import warnings
import numpy as np
import pandas as pd
import scipy.stats as st
import plotly.express as px
import plotly.graph_objects as go
from statsmodels.tsa.stattools import acf, adfuller, kpss
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")

caudales = pd.read_csv("datos/caudales.csv", parse_dates=["fecha"])
ubicaciones = pd.read_csv("datos/ubicaciones.csv", sep=";", encoding="utf-8-sig")
par = json.load(open("datos/parametros.json", encoding="utf-8"))

A_OF, B_OF = par["A_OF"], par["B_OF"]
ESTACIONES = [c for c in caudales.columns if c != "fecha"]
CORTE = pd.Timestamp("2010-01-01")
train = caudales[caudales.fecha < CORTE]

## 2.1 Calidad de los datos

El registro consolidado abarca 31.475 días, aunque la proporción de días con dato varía de
manera considerable entre estaciones. La tabla siguiente indica, para cada una, la fecha del
primer y del último dato y el porcentaje de días sin dato sobre el periodo completo.

In [2]:
cobertura = pd.DataFrame({e: caudales.loc[caudales[e].notna(), "fecha"].agg(["min", "max"])
                          for e in ESTACIONES}).T
cobertura.columns = ["Primer dato", "Último dato"]
cobertura["Días sin dato (%)"] = (100 * caudales[ESTACIONES].isna().mean()).round(1)
cobertura.sort_values("Días sin dato (%)")

,Primer dato,Último dato,Días sin dato (%)
calamar,1940-07-23,2026-09-24,1.6
pto_salgar,1946-01-01,2026-09-24,9.8
arrancaplumas,1940-07-23,2014-12-31,18.3
el_banco,1972-10-01,2026-09-24,39.6
san_roque,1972-10-01,2025-09-13,40.4
tres_cruces,1974-10-01,2026-08-31,46.7
coyongal,1975-03-01,2026-09-04,47.9


In [3]:
d = (100 * caudales[ESTACIONES].isna().mean()).sort_values().reset_index()
d.columns = ["estacion", "faltante"]
fig = px.bar(d, x="faltante", y="estacion", orientation="h", text_auto=".1f",
             labels={"faltante": "Días sin dato (%)", "estacion": ""})
fig.update_layout(height=360, margin=dict(t=10))
fig

Calamar conserva el 98,4 % del registro. Arrancaplumas y Puerto Salgar, en el alto
Magdalena, operan desde 1940 y 1946 respectivamente, aunque la primera deja de reportar a
finales de 2014, circunstancia que explica buena parte de su 18,3 % de faltantes. Las cuatro
estaciones del bajo Magdalena carecen de dato entre el 39,6 y el 47,9 % de los días,
proporción que se debe principalmente a que iniciaron su operación entre 1972 y 1975, más de
tres décadas después que la estación objetivo.

In [4]:
anual = caudales.set_index("fecha")[ESTACIONES].notna().resample("YS").mean() * 100
fig = px.imshow(anual.T, aspect="auto", color_continuous_scale="Blues",
                x=anual.index.year, labels={"x": "Año", "y": "", "color": "% con dato"})
fig.update_layout(height=360, margin=dict(t=10))
fig

La cobertura anual confirma esa lectura. Cada estación presenta un bloque de ausencia
anterior a su instalación o posterior a su cierre y, dentro de su periodo operativo, huecos
dispersos de menor extensión, de modo que la mayor parte de los faltantes refleja la
construcción progresiva de la red de medición.

Dado que la ausencia depende principalmente de la fecha, que es una variable observada, el
mecanismo dominante se clasifica como MAR. Ese diagnóstico no descarta que los huecos
internos de cada estación dependan del propio caudal, situación que correspondería a un
mecanismo MNAR y que sesgaría los estadísticos calculados sobre los días con dato. Para
examinar esa posibilidad se comparó, dentro del periodo operativo de cada estación, el caudal
de Calamar en los días con y sin dato mediante la prueba de Mann-Whitney. El estadístico U,
dividido por el producto de los tamaños de muestra, proporciona el tamaño de efecto A, que
estima la probabilidad de que un día con dato presente mayor caudal en Calamar que un día sin
dato (Mann y Whitney, 1947; Vargha y Delaney, 2000).

In [5]:
filas = []
for e in ESTACIONES:
    if e == "calamar":
        continue
    ini, fin = caudales.loc[caudales[e].notna(), "fecha"].agg(["min", "max"])
    sub = caudales[(caudales.fecha >= ini) & (caudales.fecha <= fin)]
    con = sub.loc[sub[e].notna(), "calamar"].dropna()
    sin = sub.loc[sub[e].isna(), "calamar"].dropna()
    u, p = st.mannwhitneyu(con, sin)
    filas.append({"Estación": e, "Huecos internos": len(sin),
                  "Mediana con dato": round(con.median()), "Mediana sin dato": round(sin.median()),
                  "Efecto A": round(u / (len(con) * len(sin)), 3)})
mecanismo = pd.DataFrame(filas)
mecanismo

,Estación,Huecos internos,Mediana con dato,Mediana sin dato,Efecto A
0,arrancaplumas,1314,7062,7658,0.467
1,coyongal,2353,7235,6158,0.606
2,el_banco,700,7095,7132,0.502
3,pto_salgar,1111,7126,6190,0.618
4,san_roque,590,7104,6438,0.521
5,tres_cruces,2167,7107,7141,0.530


Un valor de A igual a 0,5 indica que el caudal de Calamar se distribuye de la misma
manera en los días con y sin dato, y un valor inferior a 0,5 indicaría huecos concentrados en
aguas altas, que es el rango donde se sitúa la descarga efectiva. El único valor inferior a
0,5 corresponde a Arrancaplumas, con 0,467, y su distancia a 0,5 resulta despreciable. En
Coyongal y Puerto Salgar los valores de 0,606 y 0,618 constituyen efectos pequeños en el
sentido opuesto, con medianas de 6.158 y 6.190 m³/s en los días sin dato frente a 7.235 y
7.126 m³/s en los días con dato, de manera que sus huecos tienden a coincidir con aguas bajas
(Vargha y Delaney, 2000).

Esa asimetría no afecta la estimación de la descarga efectiva, que se calcula exclusivamente
con la serie de Calamar, y su efecto sobre el modelo queda acotado porque los huecos se
concentran lejos de la banda de interés. En consecuencia, la imputación por mediana, ajustada
dentro del Pipeline con los datos de entrenamiento, resulta adecuada para el modelo base.

## 2.2 Auditoría de la serie de transporte

El IDEAM publica para Calamar una serie diaria de transporte de sedimentos en suspensión,
expresada en kilotoneladas por día, cuyo campo de metadatos declara que el valor se obtiene
con la curva EQ_QS-QL. Una serie calculada a partir del caudal mediante una ecuación no aporta
información independiente sobre el proceso sedimentario, por lo que se verificó su naturaleza
ajustando la relación potencial $Q_s = a\,Q^b$ por bloques de cinco años sobre el periodo de
entrenamiento. El ajuste se realizó en escala logarítmica, donde la relación se vuelve lineal,
y el coeficiente de determinación mide qué fracción de la variabilidad del transporte explica
el caudal del mismo día.

Dado que los archivos originales del IDEAM no se redistribuyen, el código del ajuste se
presenta a continuación como referencia y la tabla resultante se carga desde el archivo que
emplea el tablero.

```python
q = pd.read_csv("CALAMAR_Q_MEDIA_D.csv", parse_dates=["Fecha"])[["Fecha", "Q_MEDIA_D_m3_s"]]
s = pd.read_csv("CALAMAR_TR_KT_D_QS_D.csv", parse_dates=["Fecha"])[["Fecha", "TR_KT_D_QS_D_kt_dia"]]
par_qs = q.merge(s, on="Fecha").dropna()
par_qs.columns = ["fecha", "Q", "Qs"]
par_qs = par_qs[(par_qs.fecha < CORTE) & (par_qs.Q > 0) & (par_qs.Qs > 0)]

filas = []
for ini, sub in par_qs.groupby(par_qs.fecha.dt.year // 5 * 5):
    if len(sub) < 150:
        continue
    b, loga, r, _, _ = st.linregress(np.log10(sub.Q), np.log10(sub.Qs))
    filas.append({"Bloque": f"{ini}-{ini + 4}", "Días": len(sub),
                  "Exponente b": round(b, 4), "R²": round(r ** 2, 6)})
pd.DataFrame(filas)
```

In [6]:
auditoria = pd.read_csv("datos/auditoria_transporte.csv")
auditoria

,Bloque,Días,Exponente b,R²
0,1970-1974,1096,1.4400,1.000000
1,1975-1979,1826,1.4400,1.000000
2,1980-1984,1827,1.4400,1.000000
3,1985-1989,1826,1.9401,0.960788
4,1990-1994,1826,2.0190,1.000000
5,1995-1999,1446,2.0163,0.999882
6,2000-2004,1824,1.4888,0.999739
7,2005-2009,1704,1.4993,0.997645


Seis de los ocho bloques superan un R² de 0,999, y los tres comprendidos entre 1970 y
1984 alcanzan el valor de 1 con un mismo exponente de 1,44. El exponente asciende a 1,94 en
el quinquenio 1985-1989, se mantiene cerca de 2,02 durante la década de 1990 y desciende a
1,49 y 1,50 a partir del año 2000. Los dos bloques con R² inferior a 0,999, 1985-1989 y
2005-2009, presentan exponentes intermedios entre los de sus vecinos, lo que sugiere que
mezclan días calculados con dos ecuaciones distintas.

La relación resulta exacta dentro de cada periodo y el IDEAM modificó la ecuación varias
veces, de modo que la serie publicada constituye una transformación determinista del caudal.
Incluirla como predictor introduciría el caudal por segunda vez, y emplearla para definir el
objetivo reduciría la tarea a la predicción de una fórmula cuya forma cambió con el tiempo.
Por esa razón se excluyó de ambos usos, y la curva vigente, con $a = 0{,}000569$ y
$b = 1{,}4993$, se utilizó únicamente para ponderar las clases de caudal en el análisis
magnitud-frecuencia de la sección siguiente.

## 2.3 Descarga efectiva y variable objetivo

El análisis magnitud-frecuencia se aplicó a la serie de Calamar del periodo de
entrenamiento. El rango de caudales observado se dividió en 25 clases de igual amplitud
logarítmica, se contó el número de días que el río permaneció en cada clase y se estimó la
carga asociada multiplicando esa frecuencia por el transporte que la curva oficial asigna al
centro geométrico de la clase (Andrews, 1980). La descarga efectiva corresponde al centro de
la clase cuya carga acumulada resulta máxima. Dado que el número de clases y la curva
empleada pueden desplazar la posición del máximo (Nash, 1994), el resultado se contrastó con
las cuatro ecuaciones históricas del IDEAM, que condujeron a la misma clase modal.

In [7]:
serie = train["calamar"].dropna()
bordes = np.logspace(np.log10(serie.min()), np.log10(serie.max()), 26)
frec, _ = np.histogram(serie, bins=bordes)
centros = np.sqrt(bordes[:-1] * bordes[1:])
carga = frec * A_OF * centros ** B_OF

i = carga.argmax()
Q_EF = centros[i]
LIM = (Q_EF * 0.85, Q_EF * 1.15)

print(f"Clase modal: {bordes[i]:,.0f} - {bordes[i + 1]:,.0f} m³/s")
print(f"Descarga efectiva: {Q_EF:,.0f} m³/s")
print(f"Relación con el caudal medio: {Q_EF / serie.mean():.2f}")
print(f"Excedencia: {(serie >= Q_EF).mean():.1%} de los días")
print(f"Fracción de la carga en la clase modal: {carga[i] / carga.sum():.1%}")
print(f"Fracción de los días en la clase modal: {frec[i] / frec.sum():.1%}")

Clase modal: 9,486 - 10,446 m³/s
Descarga efectiva: 9,954 m³/s
Relación con el caudal medio: 1.38
Excedencia: 15.7% de los días
Fracción de la carga en la clase modal: 14.2%
Fracción de los días en la clase modal: 9.2%


In [8]:
fig = go.Figure()
fig.add_trace(go.Bar(x=centros, y=100 * frec / frec.sum(), name="Frecuencia (% de días)",
                     marker_color="steelblue", opacity=0.6))
fig.add_trace(go.Scatter(x=centros, y=100 * carga / carga.sum(),
                         name="Carga transportada (%)", mode="lines+markers",
                         line=dict(color="sienna", width=3)))
fig.add_vline(x=Q_EF, line_dash="dash", line_color="red",
              annotation_text=f"Q_ef = {Q_EF:,.0f} m³/s")
fig.update_layout(xaxis_title="Clase de caudal (m³/s)", yaxis_title="Porcentaje",
                  height=430, margin=dict(t=30), legend=dict(orientation="h", y=1.1))
fig

La frecuencia decrece hacia los caudales altos, mientras que la carga, que combina esa
frecuencia con el transporte diario, alcanza su máximo en la clase comprendida entre 9.486 y
10.446 m³/s. La descarga efectiva resultante, 9.954 m³/s, equivale a 1,38 veces el caudal
medio del periodo de entrenamiento y se excede en el 15,7 % de los días. La clase modal
concentra el 14,2 % de la carga estimada en el 9,2 % de los días, proporción que refleja el
desplazamiento del trabajo geomorfológico hacia caudales superiores a la media (Wolman y
Miller, 1960).

In [9]:
caudales["y"] = caudales["calamar"].between(*LIM).astype(float).shift(-15)
ytr = caudales.loc[caudales.fecha < CORTE, "y"].dropna()

print(f"Banda: {LIM[0]:,.0f} - {LIM[1]:,.0f} m³/s")
print(f"Prevalencia en entrenamiento: {ytr.mean():.1%}")
print(f"Desbalance: {(1 - ytr.mean()) / ytr.mean():.1f} a 1")

Banda: 8,461 - 11,448 m³/s
Prevalencia en entrenamiento: 25.7%
Desbalance: 2.9 a 1


La variable objetivo toma el valor 1 cuando el caudal del día $t+15$ cae dentro de la
banda de ±15 % alrededor de la descarga efectiva, entre 8.461 y 11.448 m³/s. Se prefirió la
banda a un criterio de excedencia porque la descarga efectiva se refiere a un rango de
caudales en torno al máximo de transporte, y los caudales muy superiores a ese rango, aunque
movilizan más sedimento por día, ocurren con una frecuencia que los aparta del régimen
dominante. En el periodo de entrenamiento la clase positiva representa el 25,7 % de los días,
con un desbalance de 2,9 a 1. Con esa proporción, un clasificador que predijera siempre la
clase negativa alcanzaría un accuracy cercano al 74 % sin capacidad predictiva alguna, de
modo que la evaluación prioriza el F1 y el área bajo la curva precisión-exhaustividad, que se
centran en la clase minoritaria (Saito y Rehmsmeier, 2015).

## 2.4 Series de caudal y régimen hidrológico

El tablero incluye un selector que permite elegir una o varias estaciones y muestra sus
caudales promediados por mes, con la banda de descarga efectiva sombreada. Esa figura se
actualiza mediante una función de retorno que se describe en el capítulo 4. La versión
estática que se presenta a continuación muestra la estación objetivo junto con Coyongal, la
estación del bajo Magdalena con mayor correlación con ella.

In [10]:
d = caudales.set_index("fecha")[["calamar", "coyongal"]].resample("MS").mean().reset_index()
fig = px.line(d, x="fecha", y=["calamar", "coyongal"],
              labels={"value": "Caudal (m³/s)", "fecha": "Fecha", "variable": "Estación"})
fig.add_hrect(y0=LIM[0], y1=LIM[1], fillcolor="red", opacity=0.12, line_width=0)
fig

In [11]:
anios = caudales.groupby(caudales.fecha.dt.year)["calamar"]
en_banda = anios.apply(lambda s: s.between(*LIM).any())
print(f"Años con al menos un día en la banda: {en_banda.mean():.0%}")

Años con al menos un día en la banda: 93%


La serie de Calamar describe un ciclo anual marcado que la lleva a atravesar la banda de
descarga efectiva en la gran mayoría de los años del registro, de modo que la clase positiva
aparece de forma recurrente y no se concentra en un periodo particular. Coyongal reproduce las
mismas oscilaciones con un caudal inferior, consistente con su posición aguas arriba de la
estación objetivo y antes de la incorporación de algunos de los aportes del bajo Magdalena.

In [12]:
d = caudales.assign(mes=caudales.fecha.dt.month)
fig = px.box(d, x="mes", y="calamar", labels={"mes": "Mes", "calamar": "Caudal (m³/s)"})
fig.add_hline(y=Q_EF, line_dash="dash", line_color="red",
              annotation_text=f"Q_ef = {Q_EF:,.0f} m³/s")
fig.update_layout(height=400, margin=dict(t=30))
fig

In [13]:
d.groupby("mes")["calamar"].median().round()

mes
1     6712.0
2     4182.0
3     3903.0
4     4875.0
5     6782.0
6     8120.0
7     7874.0
8     6940.0
9     7188.0
10    8258.0
11    9697.0
12    9833.0
Name: calamar, dtype: float64

El régimen es bimodal. La mediana mensual desciende a 3.903 m³/s en marzo, alcanza un
máximo secundario de 8.120 m³/s en junio y asciende a 9.833 m³/s en diciembre, valor que se
aproxima a la descarga efectiva. Ese patrón se asocia al doble paso de la Zona de
Convergencia Intertropical sobre la cuenca, y su consecuencia para la tarea consiste en que
la clase positiva se concentra en noviembre y diciembre y, en menor medida, en el máximo de
mitad de año. El modelo incorpora la posición dentro del ciclo mediante componentes seno y
coseno de periodo anual y semianual, que representan ambos máximos sin introducir una
variable categórica por mes.

In [14]:
fig = go.Figure()
for e in ESTACIONES:
    s = caudales[e].dropna().sort_values(ascending=False).values
    p = 100 * (pd.Series(range(1, len(s) + 1)) / (len(s) + 1))
    paso = max(1, len(s) // 600)
    fig.add_trace(go.Scatter(x=p[::paso], y=s[::paso], name=e, mode="lines"))
fig.add_hrect(y0=LIM[0], y1=LIM[1], fillcolor="red", opacity=0.12, line_width=0)
fig.update_layout(xaxis_title="Probabilidad de excedencia (%)",
                  yaxis_title="Caudal (m³/s)", yaxis_type="log",
                  height=400, margin=dict(t=30))
fig

In [15]:
pd.DataFrame({"Mediana (m³/s)": train[ESTACIONES].median().round(),
              "Asimetría": train[ESTACIONES].skew().round(2)}).sort_values("Mediana (m³/s)")

,Mediana (m³/s),Asimetría
san_roque,462.0,0.68
arrancaplumas,1167.0,1.24
pto_salgar,1440.0,1.12
tres_cruces,2502.0,0.21
el_banco,4020.0,0.59
coyongal,4977.0,-0.13
calamar,7078.0,0.26


Calamar presenta los caudales más altos de la red en todo el rango de excedencias, con una
mediana de 7.078 m³/s en el periodo de entrenamiento, seguida de Coyongal y El Banco. El orden
de las curvas no sigue estrictamente la posición a lo largo del cauce, puesto que San Roque
registra los caudales más bajos, con una mediana de 462 m³/s, magnitud muy inferior a la de
las estaciones vecinas que indica que mide solo una fracción del caudal que llega a Calamar.
Las estaciones del alto Magdalena presentan asimetrías superiores a 1, propias de un régimen
con picos de crecida pronunciados, mientras que Calamar muestra una distribución casi
simétrica, con asimetría de 0,26, porque la llanura del bajo Magdalena y la depresión
momposina amortiguan los picos generados aguas arriba. Dado que la regresión logística opera sobre predictores
escalados y la distribución de la estación objetivo no presenta una cola pronunciada, no se
aplicó transformación logarítmica.

## 2.5 Dependencia temporal

La dependencia temporal de la serie objetivo determina el diseño de la validación. Se
calcularon la función de autocorrelación hasta un rezago de noventa días y las pruebas de
Dickey-Fuller aumentada y KPSS, cuyas hipótesis nulas son opuestas, de manera que su lectura
conjunta permite distinguir una serie estacionaria de una con raíz unitaria (Dickey y Fuller,
1979; Kwiatkowski et al., 1992; Seabold y Perktold, 2010).

In [16]:
s = train["calamar"].dropna()
ac = acf(s.values, nlags=90, fft=True)
p_adf = adfuller(s, autolag="AIC")[1]
p_kpss = kpss(s, regression="c", nlags="auto")[1]

print(f"ADF  (H0: raíz unitaria)  p = {p_adf:.4f}")
print(f"KPSS (H0: estacionaria)   p = {p_kpss:.2f}")
print(f"ACF: 1 d = {ac[1]:.3f} | 15 d = {ac[15]:.3f} | 30 d = {ac[30]:.3f}")

ADF  (H0: raíz unitaria)  p = 0.0000
KPSS (H0: estacionaria)   p = 0.10
ACF: 1 d = 0.999 | 15 d = 0.918 | 30 d = 0.749


In [17]:
fig = px.bar(x=list(range(len(ac))), y=ac,
             labels={"x": "Rezago (días)", "y": "Autocorrelación"})
for k in (15, 30):
    fig.add_vline(x=k, line_dash="dot", line_color="gray",
                  annotation_text=f"{k} d: {ac[k]:.2f}")
fig.update_layout(height=380, margin=dict(t=30), yaxis_range=[0, 1.05])
fig

La autocorrelación alcanza 0,999 a un día, 0,918 a quince y 0,749 a treinta, y decae con
lentitud a lo largo de los noventa días representados. La prueba ADF rechaza la hipótesis de
raíz unitaria (p < 0,001) y la prueba KPSS no rechaza la estacionariedad (p = 0,10, límite
superior de su tabla de valores críticos), de modo que ambas coinciden en que la serie es
estacionaria en nivel y puede modelarse sin diferenciación.

Una dependencia de esta magnitud implica que días consecutivos aportan información casi
redundante y que el tamaño efectivo de muestra resulta muy inferior al número de filas. Una
partición aleatoria situaría en el conjunto de prueba días prácticamente idénticos a otros del
conjunto de entrenamiento y produciría una estimación optimista del desempeño. Por esa razón
la partición es cronológica, con corte en 2010 y un intervalo de separación de treinta días,
superior al horizonte de predicción, y la validación cruzada emplea TimeSeriesSplit con el
mismo intervalo entre pliegues. La autocorrelación de 0,918 a quince días anticipa, además,
que la persistencia del estado actual constituirá una referencia exigente para el modelo.

## 2.6 Relación entre estaciones

La matriz de correlación de Pearson entre los caudales diarios de las siete estaciones se
calculó sobre los días en que cada par de estaciones cuenta con dato.

In [18]:
m = caudales[ESTACIONES].corr().round(2)
fig = px.imshow(m, text_auto=True, color_continuous_scale="RdYlBu_r",
                zmin=-1, zmax=1, aspect="auto")
fig.update_layout(height=430, margin=dict(t=10), coloraxis_showscale=False)
fig

La matriz revela una estructura en dos bloques. Arrancaplumas y Puerto Salgar
correlacionan entre sí con un coeficiente de 0,89 y con Calamar con valores de apenas 0,36 y
0,34, mientras que Coyongal, San Roque y El Banco alcanzan con la estación objetivo valores de
0,93, 0,87 y 0,86. Entre ambos grupos ingresan los aportes del Cauca y el Cesar y se
interpone la depresión momposina, que amortigua y redistribuye la onda de crecida (Restrepo
et al., 2006).

La correlación contemporánea subestima, sin embargo, la relación entre estaciones distantes,
porque el agua registrada aguas arriba tarda varios días en llegar a Calamar. Para estimar
ese tiempo de tránsito se calculó, con los datos de entrenamiento, la
correlación entre el caudal de cada estación desplazado entre 0 y 30 días y el caudal de
Calamar, y se adoptó el rezago que la maximiza.

In [19]:
TRANSITOS, filas = {}, []
for e in ESTACIONES:
    if e == "calamar":
        continue
    r = [train[e].shift(k).corr(train["calamar"]) for k in range(31)]
    TRANSITOS[e] = int(np.argmax(r))
    filas.append({"Estación": e, "Tránsito (días)": TRANSITOS[e],
                  "r sin rezago": round(r[0], 3), "r con rezago": round(max(r), 3)})
pd.DataFrame(filas).sort_values("Tránsito (días)")

,Estación,Tránsito (días),r sin rezago,r con rezago
1,coyongal,7,0.920,0.941
4,san_roque,10,0.878,0.917
2,el_banco,12,0.850,0.900
5,tres_cruces,16,0.723,0.799
0,arrancaplumas,20,0.359,0.467
3,pto_salgar,21,0.324,0.434


In [20]:
MAPA = {"ARRANCAPLUMAS - AUT": "arrancaplumas", "PUERTO SALGAR - AUT": "pto_salgar",
        "TRES CRUCES": "tres_cruces", "COYONGAL": "coyongal",
        "EL BANCO - AUT": "el_banco", "SAN ROQUE": "san_roque", "CALAMAR": "calamar"}
u = ubicaciones.assign(estacion=ubicaciones.Estacion.map(MAPA))
u["dias"] = u.estacion.map({**TRANSITOS, "calamar": 0})

for col in ("Altitud (m s. n. m.)", "Latitud"):
    print(f"{col:22} Pearson = {st.pearsonr(u[col], u.dias)[0]:+.2f}   "
          f"Spearman = {st.spearmanr(u[col], u.dias)[0]:+.2f}")

Altitud (m s. n. m.)   Pearson = +0.79   Spearman = +0.96
Latitud                Pearson = -0.88   Spearman = -0.86


In [21]:
fig = px.scatter(u.sort_values("dias"), x="Altitud (m s. n. m.)", y="dias", text="estacion",
                 labels={"dias": "Tiempo de tránsito (días)"})
fig.update_traces(textposition="top center", marker=dict(size=13, color="sienna"))
fig.update_layout(height=430, margin=dict(t=10))
fig

El rezago óptimo va de 7 días en Coyongal a 21 en Puerto Salgar, y en todas las
estaciones la correlación con rezago supera a la contemporánea. Dado que esos tiempos se
estimaron exclusivamente a partir de las series de caudal, su relación con la posición
geográfica de las estaciones constituye una verificación externa de su significado físico.
La correlación de rangos de Spearman entre el tiempo de tránsito y la altitud alcanza 0,96, y
la correlación de Pearson con la latitud es de −0,88, lo que indica que las estaciones más
altas y más meridionales, situadas más lejos de Calamar a lo largo del cauce, requieren más
días para transmitir su señal.

En consecuencia, cada estación entra al modelo desplazada según su tiempo de tránsito. Esa
decisión permite aprovechar la información del alto Magdalena, cuya correlación contemporánea
con Calamar es baja, y explica que las estaciones con tránsito superior a quince días
aporten información genuinamente anticipada respecto del horizonte de predicción.

## 2.7 Construcción de los predictores y auditoría de fuga

Los predictores se agrupan en cuatro conjuntos. El primero describe el estado reciente de
Calamar mediante rezagos de 1 a 60 días, medias y desviaciones móviles de 7 a 365 días y
diferencias a uno y siete días; las ventanas móviles se calculan sobre la serie desplazada un
día para excluir el valor actual de su propio resumen. El segundo incorpora cinco estaciones
aguas arriba, cada una desplazada según su tiempo de tránsito, junto con su variación en tres
días y su cociente respecto de la media del último año. Arrancaplumas se excluyó de este
conjunto porque su registro termina en 2014 y su información queda recogida por Puerto
Salgar, con la que correlaciona en 0,89. El tercero representa la posición dentro del ciclo
anual mediante los armónicos anual y semianual. El cuarto, denominado de dominio, contiene la
distancia absoluta y relativa del caudal de Calamar a la descarga efectiva, el cuadrado del
caudal y la distancia rezagada siete y quince días, variables que transforman la pertenencia a
una banda en una relación monótona que una frontera lineal puede representar.

In [22]:
PREDICTORAS = [e for e in ESTACIONES if e not in ("calamar", "arrancaplumas")]


def construir_features(d):
    x = d.sort_values("fecha").copy()
    o = "calamar"

    for k in (1, 2, 3, 5, 7, 10, 15, 30, 60):
        x[f"{o}_lag{k}"] = x[o].shift(k)

    for w in (7, 30, 90, 365):
        base = x[o].shift(1)
        x[f"{o}_media{w}"] = base.rolling(w, min_periods=max(2, w // 3)).mean()
        x[f"{o}_desv{w}"] = base.rolling(w, min_periods=max(2, w // 3)).std()

    x[f"{o}_d1"] = x[o].diff(1)
    x[f"{o}_d7"] = x[o].diff(7)

    for e in PREDICTORAS:
        k = TRANSITOS[e]
        x[f"{e}_t{k}"] = x[e].shift(k)
        x[f"{e}_d3"] = x[e].shift(k).diff(3)
        x[f"{e}_rel"] = x[e].shift(k) / x[e].shift(1).rolling(365, min_periods=120).mean()

    x["dist_qef"] = (x[o] - Q_EF).abs()
    x["dist_qef_rel"] = x["dist_qef"] / Q_EF
    x[f"{o}_cuad"] = x[o] ** 2
    for k in (7, 15):
        x[f"dist_qef_lag{k}"] = (x[f"{o}_lag{k}"] - Q_EF).abs()

    doy = x.fecha.dt.dayofyear
    x["sen_anual"] = np.sin(2 * np.pi * doy / 365.25)
    x["cos_anual"] = np.cos(2 * np.pi * doy / 365.25)
    x["sen_semi"] = np.sin(4 * np.pi * doy / 365.25)
    x["cos_semi"] = np.cos(4 * np.pi * doy / 365.25)
    return x


dfx = construir_features(caudales)

BASE_COLS = ([c for c in dfx.columns if c.startswith("calamar_") and not c.endswith("_cuad")]
             + [c for e in PREDICTORAS for c in dfx.columns if c.startswith(e + "_")]
             + ["sen_anual", "cos_anual", "sen_semi", "cos_semi"])
DOM_COLS = ["dist_qef", "dist_qef_rel", "calamar_cuad", "dist_qef_lag7", "dist_qef_lag15"]
TODAS = BASE_COLS + DOM_COLS
print(f"{len(TODAS)} predictores: {len(BASE_COLS)} base y {len(DOM_COLS)} de dominio")

43 predictores: 38 base y 5 de dominio


Todos los predictores emplean información disponible hasta el día $t$, mientras que el
objetivo se sitúa en $t+15$. Como control adicional se calculó el área bajo la curva ROC de
cada predictor por separado sobre el periodo de entrenamiento, dado que un valor cercano a 1
delataría una variable derivada del objetivo. El área se reporta como el máximo entre AUC y
1 − AUC, porque una variable inversamente relacionada con la clase positiva resulta igual de
informativa.

In [23]:
dd = dfx[dfx.fecha < CORTE].dropna(subset=["y"])
filas = []
for c in TODAS:
    sub = dd[[c, "y"]].dropna()
    auc = roc_auc_score(sub.y, sub[c])
    filas.append({"Variable": c, "AUC univariado": round(max(auc, 1 - auc), 4)})
fuga = pd.DataFrame(filas).sort_values("AUC univariado", ascending=False).head(10)
fuga

,Variable,AUC univariado
39,dist_qef_rel,0.9328
38,dist_qef,0.9328
40,calamar_cuad,0.8892
0,calamar_lag1,0.8847
41,dist_qef_lag7,0.8844
1,calamar_lag2,0.8799
2,calamar_lag3,0.8751
9,calamar_media7,0.8708
19,coyongal_t7,0.8668
3,calamar_lag5,0.8651


Ninguna variable alcanza 0,95, valor a partir del cual un predictor individual separaría
las clases de forma casi perfecta y haría sospechar de una filtración del objetivo. Las dos
variables con mayor AUC, la distancia absoluta y relativa a la descarga efectiva, llegan a
0,933 por construcción, puesto que miden la cercanía del caudal de hoy al centro de la banda.
Su poder predictivo decae con el horizonte, comportamiento que se espera de una variable
legítima y que se examina en el capítulo 3. Las variables siguientes corresponden al cuadrado
del caudal, a los rezagos cortos de Calamar y a Coyongal desplazada siete días, todas ellas
coherentes con la elevada autocorrelación de la serie.

La matriz de predictores y los parámetros que requiere el capítulo siguiente se guardan en la
carpeta de datos.

In [24]:
dfx[["fecha", "y", "calamar"] + TODAS].to_pickle("datos/features.pkl")
json.dump({"LIM": LIM, "BASE_COLS": BASE_COLS, "DOM_COLS": DOM_COLS},
          open("datos/predictores.json", "w", encoding="utf-8"), indent=2)